
# Food-101 Image Classification Using Deep Learning

**Project ID:** PRAICP-1009-101  
**Project Type:** AI / Computer Vision / Deep Learning  
**Dataset:** Food-101

## 1. Business Understanding

### Problem Statement
Develop a machine learning model that can classify the type of food from an input image.

### Project Objective
Build a deep learning image classification model capable of identifying food images across 101 different food categories.

### Proposed Methodology
1. Dataset loading and exploration
2. Image preprocessing
3. CNN model development
4. Transfer learning using EfficientNetB0
5. Model evaluation and comparison
6. Final food image prediction

### Expected Outcome
A trained image classification model that predicts the food category of an unseen image.


In [ ]:

import tensorflow as tf
import sys

print("Python Version:", sys.version)
print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices("GPU"))


Python Version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
TensorFlow Version: 2.21.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:

!nvidia-smi


Thu Oct 15 17:42:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----


## 2. Dataset Loading

The Food-101 dataset contains images belonging to 101 different food categories.

The dataset is stored as a ZIP file in Google Drive.

### Steps
1. Mount Google Drive.
2. Locate the ZIP dataset.
3. Extract the dataset into the Colab runtime.
4. Verify the extracted files and folders.


In [ ]:

import zipfile
import os

zip_path = "/content/drive/MyDrive/PRAICP_1009_101_Food_classify/food-101.zip"
extract_path = "/content/food101"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print("Extracted folders:", os.listdir(extract_path))


Dataset extracted successfully!
Extracted folders: ['lobster_roll_sandwich', 'tuna_tartare', 'crab_cakes', 'spaghetti_bolognese', 'pizza', 'oysters', 'baklava', 'club_sandwich', 'frozen_yogurt', 'apple_pie', 'caesar_salad', 'beet_salad', 'lobster_bisque', 'tacos', 'caprese_salad', 'chocolate_cake', 'ravioli', 'chicken_quesadilla', 'paella', 'cheese_plate', 'seaweed_salad', 'falafel', 'baby_back_ribs', 'bibimbap', 'hamburger', 'filet_mignon', 'nachos', 'macaroni_and_cheese', 'tiramisu', 'edamame', 'chicken_wings', 'lasagna', 'escargots', 'beef_carpaccio', 'poutine', 'macarons', 'deviled_eggs', 'prime_rib', 'ceviche', 'french_onion_soup', 'ice_cream', 'sashimi', 'bread_pudding', 'fried_rice', 'shrimp_and_grits', 'donuts', 'cannoli', 'scallops', 'hummus', 'churros', 'breakfast_burrito', 'chocolate_mousse', 'dumplings', 'spaghetti_carbonara', 'hot_and_sour_soup', 'beef_tartare', 'steak', 'grilled_salmon', 'sushi', 'garlic_bread', 'chicken_curry', 'bruschetta', 'french_toast', 'samosa', 'fi

In [ ]:

print("Extracted dataset contents:")

for item in os.listdir(extract_path):
    print(item)


Extracted dataset contents:
lobster_roll_sandwich
tuna_tartare
crab_cakes
spaghetti_bolognese
pizza
oysters
baklava
club_sandwich
frozen_yogurt
apple_pie
caesar_salad
beet_salad
lobster_bisque
tacos
caprese_salad
chocolate_cake
ravioli
chicken_quesadilla
paella
cheese_plate
seaweed_salad
falafel
baby_back_ribs
bibimbap
hamburger
filet_mignon
nachos
macaroni_and_cheese
tiramisu
edamame
chicken_wings
lasagna
escargots
beef_carpaccio
poutine
macarons
deviled_eggs
prime_rib
ceviche
french_onion_soup
ice_cream
sashimi
bread_pudding
fried_rice
shrimp_and_grits
donuts
cannoli
scallops
hummus
churros
breakfast_burrito
chocolate_mousse
dumplings
spaghetti_carbonara
hot_and_sour_soup
beef_tartare
steak
grilled_salmon
sushi
garlic_bread
chicken_curry
bruschetta
french_toast
samosa
fish_and_chips
gyoza
fried_calamari
takoyaki
pho
croque_madame
spring_rolls
panna_cotta
beignets
mussels
guacamole
red_velvet_cake
cup_cakes
omelette
miso_soup
creme_brulee
waffles
clam_chowder
french_fries
cheesecake
f


## 3. Exploratory Data Analysis (EDA)

### Objective
To understand the Food-101 dataset before training the deep learning model.

### EDA Tasks
1. Identify the total number of food classes.
2. Count the total number of images.
3. Check the number of images in each category.
4. Identify missing or empty categories.
5. Visualize sample food images.
6. Analyze image dimensions.

### Expected Outcome
Understand the dataset structure, class distribution, and image characteristics before preprocessing.


In [ ]:

import os
import pandas as pd

dataset_path = "/content/food101"

food_classes = sorted([
    folder for folder in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, folder))
])

class_counts = {}

for food in food_classes:
    folder_path = os.path.join(dataset_path, food)

    images = [
        file for file in os.listdir(folder_path)
        if file.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    class_counts[food] = len(images)

df_counts = pd.DataFrame(
    class_counts.items(),
    columns=["Food_Class", "Image_Count"]
)

print("Total Food Classes:", len(food_classes))
print("Total Images:", df_counts["Image_Count"].sum())
print("Minimum Images:", df_counts["Image_Count"].min())
print("Maximum Images:", df_counts["Image_Count"].max())

display(df_counts.head(10))


Total Food Classes: 101
Total Images: 101000
Minimum Images: 1000
Maximum Images: 1000


,Food_Class,Image_Count
0,apple_pie,1000
1,baby_back_ribs,1000
2,baklava,1000
3,beef_carpaccio,1000
4,beef_tartare,1000
5,beet_salad,1000
6,beignets,1000
7,bibimbap,1000
8,bread_pudding,1000
9,breakfast_burrito,1000



### EDA Conclusion

- The dataset contains 101,000 food images across 101 categories.
- Each category contains exactly 1,000 images.
- The dataset is perfectly balanced.
- Class balancing techniques such as oversampling or undersampling are not required.
- The dataset is ready for image visualization, preprocessing, and model development.


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image
import os
import random

random.seed(42)

sample_classes = random.sample(food_classes, 9)

plt.figure(figsize=(12, 12))

for i, food in enumerate(sample_classes):
    folder_path = os.path.join(dataset_path, food)

    image_files = [
        f for f in os.listdir(folder_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    image_name = random.choice(image_files)
    image_path = os.path.join(folder_path, image_name)

    img = Image.open(image_path)

    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title(food.replace("_", " ").title())
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

image_sizes = []

for food in sample_classes:
    folder_path = os.path.join(dataset_path, food)

    image_name = next(
        f for f in os.listdir(folder_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    )

    image_path = os.path.join(folder_path, image_name)

    with Image.open(image_path) as img:
        image_sizes.append(img.size)

print("Sample Image Dimensions:")
print(image_sizes)


Sample Image Dimensions:
[(512, 512), (512, 384), (382, 512), (512, 512), (512, 512), (512, 512), (512, 384), (512, 512), (384, 512)]



## 4. Data Preprocessing

### Objective
Prepare the Food-101 images for deep learning model training.

### Preprocessing Steps
1. Split the dataset into training (70%), validation (15%), and testing (15%).
2. Resize images to 224 × 224 pixels.
3. Convert images to RGB format.
4. Apply model-appropriate pixel preprocessing.
5. Shuffle training data.
6. Use batch loading to reduce memory consumption.

### Data Leakage Prevention
The test dataset will remain separate from training and validation and will only be used for final model evaluation.


In [ ]:

import os
import pandas as pd
from sklearn.model_selection import train_test_split

image_paths = []
labels = []

for food in food_classes:
    folder_path = os.path.join(dataset_path, food)

    for filename in os.listdir(folder_path):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths.append(os.path.join(folder_path, filename))
            labels.append(food)

df_images = pd.DataFrame({
    "image_path": image_paths,
    "label": labels
})

train_df, temp_df = train_test_split(
    df_images,
    test_size=0.30,
    stratify=df_images["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Training Images:", len(train_df))
print("Validation Images:", len(val_df))
print("Testing Images:", len(test_df))


Training Images: 70700
Validation Images: 15150
Testing Images: 15150


In [ ]:
df_images.head()

,image_path,label
0,/content/food101/apple_pie/812047.jpg,apple_pie
1,/content/food101/apple_pie/2598068.jpg,apple_pie
2,/content/food101/apple_pie/591398.jpg,apple_pie
3,/content/food101/apple_pie/2084963.jpg,apple_pie
4,/content/food101/apple_pie/532974.jpg,apple_pie


In [ ]:

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

print("\nTraining class counts:")
print(train_df["label"].value_counts().head())

print("\nValidation class counts:")
print(val_df["label"].value_counts().head())

print("\nTesting class counts:")
print(test_df["label"].value_counts().head())


Training: (70700, 2)
Validation: (15150, 2)
Testing: (15150, 2)

Training class counts:
label
chicken_wings    700
gyoza            700
beignets         700
clam_chowder     700
escargots        700
Name: count, dtype: int64

Validation class counts:
label
spaghetti_bolognese    150
caesar_salad           150
dumplings              150
fried_rice             150
eggs_benedict          150
Name: count, dtype: int64

Testing class counts:
label
caprese_salad    150
ravioli          150
beet_salad       150
tiramisu         150
garlic_bread     150
Name: count, dtype: int64



## 5. TensorFlow Data Pipeline

### Objective
Convert the training, validation, and testing datasets into TensorFlow datasets suitable for deep learning.

### Methodology
1. Encode food category labels into numerical values.
2. Load images using TensorFlow.
3. Resize images to 224 × 224 pixels.
4. Convert images into RGB tensors.
5. Normalize pixel values for the baseline CNN.
6. Create batches for efficient training.
7. Enable prefetching to improve data loading performance.

### Expected Outcome
Three TensorFlow datasets ready for CNN model development and evaluation.


In [ ]:

import tensorflow as tf

class_names = sorted(train_df["label"].unique())

class_to_index = {
    name: index
    for index, name in enumerate(class_names)
}

train_df["label_id"] = train_df["label"].map(class_to_index)
val_df["label_id"] = val_df["label"].map(class_to_index)
test_df["label_id"] = test_df["label"].map(class_to_index)

num_classes = len(class_names)

print("Number of Classes:", num_classes)
print("First 5 Classes:", class_names[:5])
print(train_df[["label", "label_id"]].head())


Number of Classes: 101
First 5 Classes: ['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare']
               label  label_id
20274  chicken_wings        20
52529          gyoza        52
6444        beignets         6
24917   clam_chowder        24
35440      escargots        35


In [ ]:

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(image_path, label):

    image = tf.io.read_file(image_path)

    image = tf.io.decode_jpeg(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    image = tf.cast(image, tf.float32)

    return image, label


In [ ]:

def create_dataset(df, training=False):

    paths = df["image_path"].values
    labels = df["label_id"].values

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(df),
            seed=42
        )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


train_ds = create_dataset(train_df, training=True)
val_ds = create_dataset(val_df)
test_ds = create_dataset(test_df)

print("TensorFlow datasets created successfully!")


TensorFlow datasets created successfully!


In [ ]:

for images, labels in train_ds.take(1):

    print("Image Batch Shape:", images.shape)
    print("Label Batch Shape:", labels.shape)
    print("Pixel Range:",
          tf.reduce_min(images).numpy(),
          tf.reduce_max(images).numpy())


Image Batch Shape: (32, 224, 224, 3)
Label Batch Shape: (32,)
Pixel Range: 0.0 255.0



## 6. Baseline CNN Model Development

### Objective
Build a Convolutional Neural Network (CNN) to classify food images into 101 categories.

### Model Architecture
1. Input layer: 224 × 224 × 3
2. Pixel normalization
3. Convolutional layers for feature extraction
4. MaxPooling layers for dimensionality reduction
5. GlobalAveragePooling layer
6. Dense layer for classification
7. Dropout layer to reduce overfitting
8. Output layer with 101 neurons and Softmax activation

### Training Configuration
- Optimizer: Adam
- Loss Function: Sparse Categorical Crossentropy
- Evaluation Metric: Accuracy
- Initial Epochs: 5
- Batch Size: 32

### Expected Outcome
Establish a baseline CNN performance for comparison with the transfer learning model.


In [ ]:

from tensorflow.keras import layers, models

cnn_model = models.Sequential([

    layers.Input(shape=(224, 224, 3)),

    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(256, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.3),

    layers.Dense(101, activation="softmax")

])

cnn_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_2 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 101)            │        13,029 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 434,341 (1.66 MB)

 Trainable params: 434,341 (1.66 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("CNN Model Compiled Successfully!")


CNN Model Compiled Successfully!


In [ ]:

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history_cnn = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop]
)


Epoch 1/10
2210/2210 ━━━━━━━━━━━━━━━━━━━━ 181s 79ms/step - accuracy: 0.0196 - loss: 4.5270 - val_accuracy: 0.0401 - val_loss: 4.3532
Epoch 2/10
2210/2210 ━━━━━━━━━━━━━━━━━━━━ 151s 68ms/step - accuracy: 0.0506 - loss: 4.2486 - val_accuracy: 0.0745 - val_loss: 4.0673
Epoch 3/10
2210/2210 ━━━━━━━━━━━━━━━━━━━━ 152s 69ms/step - accuracy: 0.0879 - loss: 3.9705 - val_accuracy: 0.1334 - val_loss: 3.7238
Epoch 4/10
2210/2210 ━━━━━━━━━━━━━━━━━━━━ 221s 78ms/step - accuracy: 0.1300 - loss: 3.7274 - val_accuracy: 0.1670 - val_loss: 3.5276
Epoch 5/10
2210/2210 ━━━━━━━━━━━━━━━━━━━━ 172s 78ms/step - accuracy: 0.1626 - loss: 3.5460 - val_accuracy: 0.1991 - val_loss: 3.3621
Epoch 6/10
2210/2210 ━━━━━━━━━━━━━━━━━━━━ 183s 69ms/step - accuracy: 0.1913 - loss: 3.3958 - val_accuracy: 0.2142 - val_loss: 3.2787
Epoch 7/10
2210/2210 ━━━━━━━━━━━━━━━━━━━━ 172s 78ms/step - accuracy: 0.2171 - loss: 3.2635 - val_accuracy: 0.2424 - val_loss: 3.1452
Epoch 8/10
2210/2210 ━━━━━━━━━━━━━━━━━━━━ 153s 69ms/step - accuracy: 


## 7. Baseline CNN Training Results

### Model Configuration
- Architecture: Custom Convolutional Neural Network
- Hidden Dense Layer: 256 neurons
- Epochs: 10
- Batch Size: 32
- Optimizer: Adam
- Loss Function: Sparse Categorical Crossentropy

### Training Results
- Final Training Accuracy: 27.68%
- Final Validation Accuracy: 29.76%
- Final Training Loss: 2.9503
- Final Validation Loss: 2.8823

### Observations
- Training and validation accuracy improved consistently.
- Both training and validation losses decreased.
- No clear evidence of overfitting was observed.
- The model has not yet achieved strong classification performance.

### Conclusion
The custom CNN establishes an initial baseline for the Food-101 classification task. Transfer learning with EfficientNetB0 will be explored to improve classification performance.


In [ ]:

import os

save_dir = "/content/drive/MyDrive/Food101_Project"
os.makedirs(save_dir, exist_ok=True)

cnn_model.save(
    os.path.join(save_dir, "food101_baseline_cnn.keras")
)

print("Baseline CNN model saved successfully!")


Baseline CNN model saved successfully!



## 8. Transfer Learning Using EfficientNetB0

### Objective
Improve Food-101 image classification performance using a pretrained EfficientNetB0 model.

### Why Transfer Learning?
The baseline CNN achieved 29.76% validation accuracy after 10 epochs.

Training a CNN from scratch requires learning all image features from the beginning.

EfficientNetB0 is pretrained on ImageNet and can reuse learned visual features such as edges, textures, shapes, and object patterns.

### Methodology
1. Load EfficientNetB0 with ImageNet weights.
2. Remove the original classification layer.
3. Freeze the pretrained base model.
4. Add a new classification head for 101 food categories.
5. Compile the model using Adam optimizer.
6. Train and validate the model.
7. Compare performance with the baseline CNN.
8. Optionally fine-tune selected pretrained layers.

### Expected Outcome
Achieve better classification performance than the baseline CNN.


In [ ]:

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

print("EfficientNetB0 loaded successfully!")


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
EfficientNetB0 loaded successfully!


In [ ]:
num_classes = 101
inputs = layers.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(256, activation="relu")(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax"
)(x)

efficientnet_model = Model(
    inputs=inputs,
    outputs=outputs
)

efficientnet_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 101)            │        25,957 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,403,464 (16.80 MB)

 Trainable params: 353,893 (1.35 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
print("Classes:", num_classes)
print("Training Dataset:", train_ds)
print("Validation Dataset:", val_ds)

Classes: 101
Training Dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>
Validation Dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>


In [ ]:

efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("EfficientNetB0 compiled successfully!")


EfficientNetB0 compiled successfully!


In [ ]:

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

checkpoint = ModelCheckpoint(
    "/content/drive/MyDrive/Food101_Project/best_efficientnet.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stop_eff = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history_eff = efficientnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[
        checkpoint,
        early_stop_eff
    ]
)


Epoch 1/5
2210/2210 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.4214 - loss: 2.3919
Epoch 1: val_accuracy improved from None to 0.63987, saving model to /content/drive/MyDrive/Food101_Project/best_efficientnet.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Food101_Project/best_efficientnet.keras
2210/2210 ━━━━━━━━━━━━━━━━━━━━ 237s 93ms/step - accuracy: 0.5141 - loss: 1.9244 - val_accuracy: 0.6399 - val_loss: 1.3777
Epoch 2/5
2209/2210 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.6163 - loss: 1.4452
Epoch 2: val_accuracy improved from 0.63987 to 0.65789, saving model to /content/drive/MyDrive/Food101_Project/best_efficientnet.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Food101_Project/best_efficientnet.keras
2210/2210 ━━━━━━━━━━━━━━━━━━━━ 174s 79ms/step - accuracy: 0.6162 - loss: 1.4501 - val_accuracy: 0.6579 - val_loss: 1.3022
Epoch 3/5
2210/2210 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.6516 - loss: 1.3024
Epoch 3: val_accuracy improve